In [1]:
library(Matrix)
library(scran)
library(Rtsne)
library(irlba)
library(cowplot)

source("/rds/project/bg200/rds-bg200-hphi-gottgens/users/bt392/mouse/Mixl1_KO/core_functions.R")
load_data()

nPC = 50

Loading required package: SingleCellExperiment

Loading required package: SummarizedExperiment

Loading required package: GenomicRanges

Loading required package: stats4

Loading required package: BiocGenerics

Loading required package: parallel


Attaching package: ‘BiocGenerics’


The following objects are masked from ‘package:parallel’:

    clusterApply, clusterApplyLB, clusterCall, clusterEvalQ,
    clusterExport, clusterMap, parApply, parCapply, parLapply,
    parLapplyLB, parRapply, parSapply, parSapplyLB


The following object is masked from ‘package:Matrix’:

    which


The following objects are masked from ‘package:stats’:

    IQR, mad, sd, var, xtabs


The following objects are masked from ‘package:base’:

    anyDuplicated, append, as.data.frame, basename, cbind, colnames,
    dirname, do.call, duplicated, eval, evalq, Filter, Find, get, grep,
    grepl, intersect, is.unsorted, lapply, Map, mapply, match, mget,
    order, paste, pmax, pmax.int, pmin, pmin.int, Position, r

In [24]:
head(meta)

,cell,barcode,sample,stage,batch
,<chr>,<chr>,<int>,<dbl>,<int>
1,cell_1,AAACCCAAGCATAGGC,1,8.5,1
2,cell_2,AAACCCAAGGGACTGT,1,8.5,1
3,cell_3,AAACCCACAAGCAATA,1,8.5,1
4,cell_4,AAACCCACATCTTCGC,1,8.5,1
5,cell_5,AAACCCAGTATCTCTT,1,8.5,1
6,cell_6,AAACCCAGTCCAATCA,1,8.5,1


In [16]:
hvgs = getHVGs(sce)

correct = doBatchCorrect(counts = logcounts(sce[hvgs,]),
                         timepoints = as.character(meta$tomato), #first correct genotypes separately
                         samples = meta$sample,
                         timepoint_order = c("FALSE", "TRUE"), #host cells first
                         sample_order = 1:4 #doesn't matter as pairwise correction
                         )


corrected = list(all = correct[match(meta$cell, rownames(correct)),])
base = prcomp_irlba(t(logcounts(scater::normalize(sce[hvgs, ]))), n = nPC)$x

saveRDS(corrected, file = "/rds/project/bg200/rds-bg200-hphi-gottgens/users/bt392/mouse/Mixl1_KO/corrected_pcas.rds")
saveRDS(base, file = "/rds/project/bg200/rds-bg200-hphi-gottgens/users/bt392/mouse/Mixl1_KO/base_pca.rds")

In [17]:
tsne_pre = Rtsne(base, pca = FALSE)$Y
tsne_post = Rtsne(corrected$all, pca = FALSE)$Y

ro = sample(nrow(base), nrow(base))




p1 = ggplot(as.data.frame(tsne_pre)[ro,], aes(x = V1, y = V2, col = factor(meta$sample)[ro])) +
  geom_point(size = 0.4) +
  scale_colour_manual(values = c("3" = "black", "1" = "red", "2" = "coral", "4" = "darkgrey")) +
  theme(legend.position = "none",
        axis.line = element_blank(),
        axis.text = element_blank(),
        axis.ticks = element_blank(),
        axis.title = element_blank()) +
  ggtitle("Pre-correction")

p2 = ggplot(as.data.frame(tsne_post)[ro,], aes(x = V1, y = V2, col = factor(meta$sample)[ro])) +
  geom_point(size = 0.4) +
  scale_colour_manual(values = c("3" = "black", "1" = "red", "2" = "coral", "4" = "darkgrey")) +
  theme(legend.position = "none",
        axis.line = element_blank(),
        axis.text = element_blank(),
        axis.ticks = element_blank(),
        axis.title = element_blank()) +
  ggtitle("Post-correction")


plot_grid(p1, p2, nrow = 2)

Loading required package: biomaRt

Warning message in library(package, lib.loc = lib.loc, character.only = TRUE, logical.return = TRUE, :
“there is no package called ‘biomaRt’”


ERROR: Error in scran::fitTrendVar(sce, use.spikes = FALSE, loess.args = list(span = 0.05)): argument "vars" is missing, with no default
